# Fake News Detection — SHAP and LIME Explainability

Applies both SHAP and LIME to the TF-IDF baseline and DistilBERT predictions on the same ISOT test set. Also identifies cases where the baseline and DistilBERT disagree, since those are the most informative cases for explanation quality.

Requires: `pip install shap lime`



## 1. Load Saved Models and Test Data


In [ ]:
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_df = pd.read_csv("test.csv")

baseline_pipeline = joblib.load("baseline.joblib")

distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained("./distilbert_fake_news_final")
distilbert_model = DistilBertForSequenceClassification.from_pretrained("./distilbert_fake_news_final")
distilbert_model.to(device)
distilbert_model.eval()

print("Test set size:", len(test_df))


Using device: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test set size: 5796


## 2. Find Prediction Disagreements

Cases where the TF-IDF baseline and DistilBERT disagree on the label — they reveal where a simple word-frequency model and a contextual transformer diverge in reasoning.


In [ ]:
MAX_LENGTH = 256

def distilbert_predict_proba(texts, batch_size=8):
    """Returns [P(real), P(fake)] for a list of texts, processed in small batches to avoid OOM."""
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = distilbert_tokenizer(
            batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = distilbert_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

# Run predictions on the full test set (batched for speed)
BATCH_SIZE = 32
all_probs = []
texts = test_df["text"].astype(str).tolist()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    probs = distilbert_predict_proba(batch)
    all_probs.append(probs)

distilbert_probs = np.vstack(all_probs)
distilbert_preds = distilbert_probs.argmax(axis=1)

baseline_preds = baseline_pipeline.predict(test_df["text"].astype(str))

test_df["baseline_pred"] = baseline_preds
test_df["distilbert_pred"] = distilbert_preds
test_df["disagreement"] = test_df["baseline_pred"] != test_df["distilbert_pred"]

print(f"Total disagreements: {test_df['disagreement'].sum()} out of {len(test_df)} ({test_df['disagreement'].mean():.1%})")


Total disagreements: 82 out of 5796 (1.4%)


In [3]:
disagreement_df = test_df[test_df["disagreement"]].copy()
print(f"Disagreement cases: {len(disagreement_df)}")
disagreement_df[["text", "label", "baseline_pred", "distilbert_pred"]].head(10)


Disagreement cases: 82


,text,label,baseline_pred,distilbert_pred
16,(Reuters) - Fourteen years after criticizing A...,real,1,0
53,This is pretty scary stuff. A federal governme...,fake,0,1
54,NEW YORK (Reuters) - Two aides in charge of ru...,real,1,0
156,"Iran, where the thought police will always hav...",fake,0,1
173,"BETHLEHEM, Pennsylvania (Reuters) - The unrave...",real,1,0
229,I appears that Hillary Clinton is really in a ...,fake,0,1
254,Thanks to the funding our GOP majority Congres...,fake,0,1
305,(Reuters) - President Donald Trump on Friday w...,real,1,0
320,California gained an embassy in Russia last we...,fake,0,1
377,Boehner s hightailing it out of DC just in tim...,fake,0,1


## 3. SHAP on TF-IDF Baseline

Uses `LinearExplainer`, which computes exact Shapley values for linear models.


In [4]:
import shap

tfidf_vectorizer = baseline_pipeline.named_steps["tfidf"]
logreg_clf = baseline_pipeline.named_steps["clf"]

# Background sample for the explainer (standard practice: a representative subset of training-like data)
background_texts = test_df["text"].astype(str).sample(100, random_state=42)
background_vectors = tfidf_vectorizer.transform(background_texts)

tfidf_explainer = shap.LinearExplainer(logreg_clf, background_vectors)

def explain_tfidf(text, top_k=10):
    vec = tfidf_vectorizer.transform([text])
    shap_values = tfidf_explainer.shap_values(vec)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    # shap_values shape: (1, n_features) for binary LinearExplainer
    values = shap_values[0] if len(shap_values.shape) == 2 else shap_values
    nonzero_idx = vec.nonzero()[1]

    contributions = [(feature_names[i], values[i]) for i in nonzero_idx]
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    return contributions[:top_k]

# Example: explain one disagreement case
example_text = disagreement_df.iloc[0]["text"]
print("Example text (truncated):", example_text[:200])
print("\nTop SHAP contributions (TF-IDF):")
for word, val in explain_tfidf(example_text):
    direction = "toward FAKE" if val > 0 else "toward REAL"
    print(f"  {word}: {val:.4f} ({direction})")


Example text (truncated): (Reuters) - Fourteen years after criticizing Augusta National Golf Club for its all-male membership policy, women’s issues expert Martha Burk has called for next year’s U.S. Women’s Open to be moved f

Top SHAP contributions (TF-IDF):
  year: -0.2576 (toward REAL)
  national: -0.1484 (toward REAL)
  trump: 0.1422 (toward FAKE)
  racist: 0.1309 (toward FAKE)
  course: 0.1142 (toward FAKE)
  women: 0.1139 (toward FAKE)
  words: 0.1025 (toward FAKE)
  golf: -0.0978 (toward REAL)
  statement: -0.0942 (toward REAL)
  tournament: -0.0827 (toward REAL)


## 4. SHAP on DistilBERT

Uses a masking-based `shap.Explainer` with a text masker, since DistilBERT is a black box — SHAP estimates contribution by systematically masking tokens and observing how the prediction changes.

In [5]:
def distilbert_predict_for_shap(texts):
    """Wrapper matching the interface shap.Explainer expects."""
    return distilbert_predict_proba(list(texts))

masker = shap.maskers.Text(distilbert_tokenizer)
distilbert_explainer = shap.Explainer(distilbert_predict_for_shap, masker)

# SHAP on transformers is slow — start with a small sample, not the full test set or even full disagreement set
sample_for_shap = disagreement_df["text"].head(5).tolist()

distilbert_shap_values = distilbert_explainer(sample_for_shap)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (694 > 512). Running this sequence through the model will result in indexing errors



PartitionExplainer explainer:  20%|██        | 1/5 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  60%|██████    | 3/5 [00:21<00:08,  4.20s/it]


PartitionExplainer explainer:  80%|████████  | 4/5 [00:27<00:05,  5.01s/it]


PartitionExplainer explainer: 100%|██████████| 5/5 [00:31<00:00,  4.73s/it]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer: 6it [00:41,  6.34s/it]                       


PartitionExplainer explainer: 6it [00:41,  8.23s/it]

In [6]:
# Inspect the first example's top contributing tokens toward the "fake" class (index 1)
def top_tokens_from_shap(shap_explanation, class_idx=1, top_k=10):
    tokens = shap_explanation.data
    values = shap_explanation.values[:, class_idx]
    pairs = list(zip(tokens, values))
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:top_k]

print("Example text (truncated):", sample_for_shap[0][:200])
print("\nTop SHAP contributions (DistilBERT, toward FAKE class):")
for token, val in top_tokens_from_shap(distilbert_shap_values[0]):
    direction = "toward FAKE" if val > 0 else "toward REAL"
    print(f"  '{token.strip()}': {val:.4f} ({direction})")


Example text (truncated): (Reuters) - Fourteen years after criticizing Augusta National Golf Club for its all-male membership policy, women’s issues expert Martha Burk has called for next year’s U.S. Women’s Open to be moved f

Top SHAP contributions (DistilBERT, toward FAKE class):
  '-': -0.0560 (toward REAL)
  'Fourteen': -0.0560 (toward REAL)
  '(': -0.0469 (toward REAL)
  'Reuters': -0.0469 (toward REAL)
  ')': -0.0469 (toward REAL)
  'Post': -0.0081 (toward REAL)
  'on': -0.0081 (toward REAL)
  'Friday': -0.0081 (toward REAL)
  'The': -0.0081 (toward REAL)
  'Huffington': -0.0081 (toward REAL)


## 5. LIME on TF-IDF Baseline


In [7]:
from lime.lime_text import LimeTextExplainer

lime_explainer = LimeTextExplainer(class_names=["real", "fake"])

def tfidf_predict_proba(texts):
    vecs = tfidf_vectorizer.transform(texts)
    return logreg_clf.predict_proba(vecs)

lime_exp_tfidf = lime_explainer.explain_instance(
    example_text, tfidf_predict_proba, num_features=10
)

print("LIME explanation (TF-IDF):")
for word, weight in lime_exp_tfidf.as_list():
    direction = "toward FAKE" if weight > 0 else "toward REAL"
    print(f"  {word}: {weight:.4f} ({direction})")


LIME explanation (TF-IDF):
  Reuters: -0.0974 (toward REAL)
  year: -0.0837 (toward REAL)
  Trump: 0.0817 (toward FAKE)
  racist: 0.0517 (toward FAKE)
  National: -0.0479 (toward REAL)
  Republican: -0.0421 (toward REAL)
  presidential: -0.0415 (toward REAL)
  course: 0.0393 (toward FAKE)
  statement: -0.0383 (toward REAL)
  America: 0.0316 (toward FAKE)


## 6. LIME on DistilBERT


In [8]:
lime_exp_distilbert = lime_explainer.explain_instance(
    sample_for_shap[0], distilbert_predict_for_shap, num_features=10, num_samples=500
)

print("LIME explanation (DistilBERT):")
for word, weight in lime_exp_distilbert.as_list():
    direction = "toward FAKE" if weight > 0 else "toward REAL"
    print(f"  {word}: {weight:.4f} ({direction})")


LIME explanation (DistilBERT):
  Reuters: -0.0000 (toward REAL)
  policy: -0.0000 (toward REAL)
  chairman: -0.0000 (toward REAL)
  Trump: -0.0000 (toward REAL)
  a: -0.0000 (toward REAL)
  years: 0.0000 (toward FAKE)
  has: 0.0000 (toward FAKE)
  an: -0.0000 (toward REAL)
  many: 0.0000 (toward FAKE)
  Miami: 0.0000 (toward FAKE)


## 7. Compare SHAP vs LIME on Disagreement Cases

Runs both explanation methods across several disagreement cases and times each, to compare quality and speed.


In [9]:
import time

comparison_rows = []
n_cases = min(5, len(disagreement_df))

for idx in range(n_cases):
    row = disagreement_df.iloc[idx]
    text = row["text"]

    # SHAP (TF-IDF) timing
    start = time.time()
    shap_tfidf_result = explain_tfidf(text, top_k=5)
    shap_tfidf_time = time.time() - start

    # LIME (TF-IDF) timing
    start = time.time()
    lime_tfidf_result = lime_explainer.explain_instance(text, tfidf_predict_proba, num_features=5)
    lime_tfidf_time = time.time() - start

    comparison_rows.append({
        "text_preview": text[:80],
        "true_label": row["label"],
        "baseline_pred": "fake" if row["baseline_pred"] == 1 else "real",
        "distilbert_pred": "fake" if row["distilbert_pred"] == 1 else "real",
        "shap_tfidf_top_words": [w for w, v in shap_tfidf_result[:5]],
        "shap_tfidf_time_sec": shap_tfidf_time,
        "lime_tfidf_top_words": [w for w, v in lime_tfidf_result.as_list()[:5]],
        "lime_tfidf_time_sec": lime_tfidf_time,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("shap_lime_disagreement_comparison.csv", index=False)
comparison_df


,text_preview,true_label,baseline_pred,distilbert_pred,shap_tfidf_top_words,shap_tfidf_time_sec,lime_tfidf_top_words,lime_tfidf_time_sec
0,(Reuters) - Fourteen years after criticizing A...,real,fake,real,"[year, national, trump, racist, course]",0.056248,"[Reuters, year, Trump, National, racist]",5.364766
1,This is pretty scary stuff. A federal governme...,fake,real,fake,"[election, state, said, presidential, dhs]",0.017959,"[said, election, DHS, presidential, Georgia]",7.495256
2,NEW YORK (Reuters) - Two aides in charge of ru...,real,fake,real,"[podesta, clinton, wikileaks, hillary, emails]",0.017552,"[said, Clinton, Reuters, Podesta, Hillary]",6.599827
3,"Iran, where the thought police will always hav...",fake,real,fake,"[spokesman, sites, press, statement, military]",0.017087,"[said, Press, spokesman, statement, Monday]",3.465902
4,"BETHLEHEM, Pennsylvania (Reuters) - The unrave...",real,fake,real,"[obama, percent, county, said, republican]",0.020107,"[Obama, percent, just, said, Republican]",10.502130


## 8. RoBERTa


In [10]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

roberta_tokenizer = RobertaTokenizerFast.from_pretrained("./roberta_fake_news_final")
roberta_model = RobertaForSequenceClassification.from_pretrained("./roberta_fake_news_final")
roberta_model.to(device)
roberta_model.eval()

def roberta_predict_proba(texts, batch_size=8):
    """Returns [P(real), P(fake)] for a list of texts, batched to avoid OOM."""
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = roberta_tokenizer(
            batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = roberta_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

def roberta_predict_for_shap(texts):
    return roberta_predict_proba(list(texts))

# Run RoBERTa predictions on the full test set
BATCH_SIZE = 32
all_probs = []
texts = test_df["text"].astype(str).tolist()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    probs = roberta_predict_proba(batch)
    all_probs.append(probs)

roberta_probs = np.vstack(all_probs)
roberta_preds = roberta_probs.argmax(axis=1)

test_df["roberta_pred"] = roberta_preds
print("RoBERTa predictions added.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa predictions added.


### 8.1 Pairwise Disagreement Across All Three Models


In [11]:
test_df["disagree_baseline_distilbert"] = test_df["baseline_pred"] != test_df["distilbert_pred"]
test_df["disagree_baseline_roberta"] = test_df["baseline_pred"] != test_df["roberta_pred"]
test_df["disagree_distilbert_roberta"] = test_df["distilbert_pred"] != test_df["roberta_pred"]
test_df["disagree_any"] = (
    test_df["disagree_baseline_distilbert"] |
    test_df["disagree_baseline_roberta"] |
    test_df["disagree_distilbert_roberta"]
)

print(f"Baseline vs DistilBERT disagreements: {test_df['disagree_baseline_distilbert'].sum()} ({test_df['disagree_baseline_distilbert'].mean():.1%})")
print(f"Baseline vs RoBERTa disagreements: {test_df['disagree_baseline_roberta'].sum()} ({test_df['disagree_baseline_roberta'].mean():.1%})")
print(f"DistilBERT vs RoBERTa disagreements: {test_df['disagree_distilbert_roberta'].sum()} ({test_df['disagree_distilbert_roberta'].mean():.1%})")
print(f"\nAny disagreement (at least one pair): {test_df['disagree_any'].sum()} ({test_df['disagree_any'].mean():.1%})")


Baseline vs DistilBERT disagreements: 82 (1.4%)
Baseline vs RoBERTa disagreements: 84 (1.4%)
DistilBERT vs RoBERTa disagreements: 6 (0.1%)

Any disagreement (at least one pair): 86 (1.5%)


### 8.2 False Positives

Cases where a model incorrectly predicts "fake" when the true label is "real" — the most relevant disagreement cases to inspect with SHAP/LIME.


In [12]:
baseline_false_positives = test_df[(test_df["label"] == "real") & (test_df["baseline_pred"] == 1)]
distilbert_false_positives = test_df[(test_df["label"] == "real") & (test_df["distilbert_pred"] == 1)]
roberta_false_positives = test_df[(test_df["label"] == "real") & (test_df["roberta_pred"] == 1)]

print(f"Baseline false positives: {len(baseline_false_positives)}")
print(f"DistilBERT false positives: {len(distilbert_false_positives)}")
print(f"RoBERTa false positives: {len(roberta_false_positives)}")


Baseline false positives: 24
DistilBERT false positives: 1
RoBERTa false positives: 6


### 8.3 SHAP on RoBERTa

Same masking-based approach used for DistilBERT.


In [13]:
roberta_masker = shap.maskers.Text(roberta_tokenizer)
roberta_explainer = shap.Explainer(roberta_predict_for_shap, roberta_masker)

# Small sample — SHAP on transformers is slow
roberta_disagreement_df = test_df[test_df["disagree_baseline_roberta"]].copy()
sample_for_roberta_shap = roberta_disagreement_df["text"].head(5).tolist()

roberta_shap_values = roberta_explainer(sample_for_roberta_shap)

print("Example text (truncated):", sample_for_roberta_shap[0][:200])
print("\nTop SHAP contributions (RoBERTa, toward FAKE class):")
for token, val in top_tokens_from_shap(roberta_shap_values[0]):
    direction = "toward FAKE" if val > 0 else "toward REAL"
    print(f"  '{token.strip()}': {val:.4f} ({direction})")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2643 > 512). Running this sequence through the model will result in indexing errors


  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  20%|██        | 1/5 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  60%|██████    | 3/5 [01:17<00:59, 29.55s/it]

...


PartitionExplainer explainer: 100%|██████████| 5/5 [02:01<00:00, 23.61s/it]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer: 6it [03:06, 38.07s/it]                       


PartitionExplainer explainer: 6it [03:06, 37.27s/it]

Example text (truncated): (Reuters) - Fourteen years after criticizing Augusta National Golf Club for its all-male membership policy, women’s issues expert Martha Burk has called for next year’s U.S. Women’s Open to be moved f

Top SHAP contributions (RoBERTa, toward FAKE class):
  '': -0.0505 (toward REAL)
  '(': -0.0505 (toward REAL)
  'R': -0.0505 (toward REAL)
  'e': -0.0505 (toward REAL)
  'u': -0.0505 (toward REAL)
  't': -0.0505 (toward REAL)
  ')': -0.0505 (toward REAL)
  'e': -0.0505 (toward REAL)
  'r': -0.0505 (toward REAL)
  's': -0.0505 (toward REAL)


### 8.4 LIME on RoBERTa


In [14]:
lime_exp_roberta = lime_explainer.explain_instance(
    sample_for_roberta_shap[0], roberta_predict_for_shap, num_features=10, num_samples=500
)

print("LIME explanation (RoBERTa):")
for word, weight in lime_exp_roberta.as_list():
    direction = "toward FAKE" if weight > 0 else "toward REAL"
    print(f"  {word}: {weight:.4f} ({direction})")


LIME explanation (RoBERTa):
  million: -0.0027 (toward REAL)
  accountability: -0.0022 (toward REAL)
  finally: -0.0020 (toward REAL)
  club: -0.0014 (toward REAL)
  basis: 0.0013 (toward FAKE)
  then: 0.0013 (toward FAKE)
  angered: 0.0008 (toward FAKE)
  Los: 0.0004 (toward FAKE)
  complications: -0.0004 (toward REAL)
  behind: 0.0002 (toward FAKE)


### 8.5 Full Three-Model Disagreement + Explanation Summary

Combines predictions and top SHAP/LIME words across all three models for each disagreement case, for a single reference table.


In [15]:
all_disagreements = test_df[test_df["disagree_any"]].copy()
print(f"Total cases with at least one model disagreeing: {len(all_disagreements)}")

summary_rows = []
n_cases = min(10, len(all_disagreements))

for idx in range(n_cases):
    row = all_disagreements.iloc[idx]
    summary_rows.append({
        "text_preview": row["text"][:100],
        "true_label": row["label"],
        "baseline_pred": "fake" if row["baseline_pred"] == 1 else "real",
        "distilbert_pred": "fake" if row["distilbert_pred"] == 1 else "real",
        "roberta_pred": "fake" if row["roberta_pred"] == 1 else "real",
        "contains_reuters": "reuters" in row["text"].lower(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("three_model_disagreement_summary.csv", index=False)
summary_df


Total cases with at least one model disagreeing: 86


,text_preview,true_label,baseline_pred,distilbert_pred,roberta_pred,contains_reuters
0,(Reuters) - Fourteen years after criticizing A...,real,fake,real,real,True
1,This is pretty scary stuff. A federal governme...,fake,real,fake,fake,False
2,NEW YORK (Reuters) - Two aides in charge of ru...,real,fake,real,real,True
3,"Iran, where the thought police will always hav...",fake,real,fake,fake,False
4,"BETHLEHEM, Pennsylvania (Reuters) - The unrave...",real,fake,real,real,True
5,I appears that Hillary Clinton is really in a ...,fake,real,fake,fake,False
6,Thanks to the funding our GOP majority Congres...,fake,real,fake,fake,False
7,(Reuters) - President Donald Trump on Friday w...,real,fake,real,real,True
8,California gained an embassy in Russia last we...,fake,real,fake,fake,False
9,Boehner s hightailing it out of DC just in tim...,fake,real,fake,fake,False


## 9. Reuters/Dateline Pattern Check

Checks for a wire-service dateline pattern: a location name followed by "(Reuters)" near the start of the article, e.g. "WASHINGTON (Reuters) -".


In [16]:
import re

def has_reuters_dateline(text, check_chars=100):
    """Checks if '(Reuters)' appears near the start of the article (typical wire-service dateline format)."""
    return bool(re.search(r"\(Reuters\)", text[:check_chars]))

def has_reuters_anywhere(text):
    return "reuters" in text.lower()

# Apply to the full test set once, reused for all breakdowns below
test_df["has_reuters_dateline"] = test_df["text"].astype(str).apply(has_reuters_dateline)
test_df["has_reuters_anywhere"] = test_df["text"].astype(str).apply(has_reuters_anywhere)

print("Reuters dateline (near start) present in full test set, by true label:")
print(test_df.groupby("label")["has_reuters_dateline"].mean())

print("\n'Reuters' anywhere in text, by true label:")
print(test_df.groupby("label")["has_reuters_anywhere"].mean())


Reuters dateline (near start) present in full test set, by true label:
label
fake    0.000000
real    0.986788
Name: has_reuters_dateline, dtype: float64

'Reuters' anywhere in text, by true label:
label
fake    0.011846
real    0.998113
Name: has_reuters_anywhere, dtype: float64


### 9.1 Reuters Pattern in False Positives — All Three Models

For each model, what fraction of its false positives (predicted fake, actually real) contain the Reuters dateline pattern — compared against the base rate in the overall "real" class.


In [17]:
results = []

for model_name, pred_col in [
    ("TF-IDF + Logistic Regression", "baseline_pred"),
    ("DistilBERT", "distilbert_pred"),
    ("RoBERTa", "roberta_pred"),
]:
    fp = test_df[(test_df["label"] == "real") & (test_df[pred_col] == 1)]
    n_fp = len(fp)
    pct_dateline = fp["has_reuters_dateline"].mean() if n_fp > 0 else float("nan")
    pct_anywhere = fp["has_reuters_anywhere"].mean() if n_fp > 0 else float("nan")

    results.append({
        "model": model_name,
        "n_false_positives": n_fp,
        "pct_with_reuters_dateline": pct_dateline,
        "pct_with_reuters_anywhere": pct_anywhere,
    })

# Base rate for comparison: how often does '(Reuters)' dateline appear in ALL real articles overall
base_rate = test_df[test_df["label"] == "real"]["has_reuters_dateline"].mean()
print(f"Base rate: {base_rate:.1%} of ALL real articles have a Reuters dateline near the start\n")

fp_summary = pd.DataFrame(results)
fp_summary.to_csv("reuters_false_positive_analysis.csv", index=False)
fp_summary


Base rate: 98.7% of ALL real articles have a Reuters dateline near the start



,model,n_false_positives,pct_with_reuters_dateline,pct_with_reuters_anywhere
0,TF-IDF + Logistic Regression,24,0.958333,1.000000
1,DistilBERT,1,0.000000,0.000000
2,RoBERTa,6,0.000000,0.333333


### 9.2 Interpretation

If false positives (real articles wrongly predicted as fake) have a **lower** rate of Reuters dateline presence than the overall real-article base rate, that's consistent with the leakage hypothesis: the model may be relying on the dateline as a strong "real" signal, and specifically fails on the real articles that happen to lack it.


In [18]:
print(f"Base rate (all real articles with Reuters dateline): {base_rate:.1%}\n")

for row in results:
    diff = row['pct_with_reuters_dateline'] - base_rate
    direction = "LOWER than base rate" if diff < 0 else "HIGHER than base rate"
    print(f"{row['model']}: {row['pct_with_reuters_dateline']:.1%} of false positives have dateline ({direction} by {abs(diff):.1%})")


Base rate (all real articles with Reuters dateline): 98.7%

TF-IDF + Logistic Regression: 95.8% of false positives have dateline (LOWER than base rate by 2.8%)
DistilBERT: 0.0% of false positives have dateline (LOWER than base rate by 98.7%)
RoBERTa: 0.0% of false positives have dateline (LOWER than base rate by 98.7%)


## 10. Robustness Check — Stripping the Reuters Dateline

Tests how much each model's performance depends on the dateline shortcut.


In [19]:
import re

def strip_dateline(text, check_chars=150):
    """Removes a leading wire-service dateline pattern, e.g. 'WASHINGTON (Reuters) - '."""
    # Pattern: optional ALL-CAPS location, then (Reuters), then a dash/hyphen separator
    pattern = r"^.{0,80}\(Reuters\)\s*[-–—]\s*"
    stripped = re.sub(pattern, "", text[:check_chars], count=1) + text[check_chars:]
    return stripped

# Quick check that the stripping actually works on a few examples
sample_real = test_df[test_df["label"] == "real"]["text"].head(3)
for t in sample_real:
    print("BEFORE:", t[:120])
    print("AFTER: ", strip_dateline(t)[:120])
    print()


BEFORE: BELGRADE (Reuters) - Turkish President Tayyip Erdogan said on Tuesday the United States should dismiss its ambassador to
AFTER:  Turkish President Tayyip Erdogan said on Tuesday the United States should dismiss its ambassador to Ankara if he took th

BEFORE: WASHINGTON (Reuters) - Several Democratic senators pressed billionaire investor Carl Icahn on Monday to clarify his role
AFTER:  Several Democratic senators pressed billionaire investor Carl Icahn on Monday to clarify his role as an adviser to Presi

BEFORE:  WASHINGTON (Reuters) - When President Barack Obama entered the White House in 2009, the federal appeals court based in 
AFTER:  When President Barack Obama entered the White House in 2009, the federal appeals court based in Virginia was known as on



In [ ]:
test_df["text_stripped"] = test_df["text"].astype(str).apply(strip_dateline)

# Confirm the dateline is actually gone
test_df["still_has_dateline"] = test_df["text_stripped"].apply(has_reuters_dateline)
print("Real articles still containing dateline after stripping:")
print(test_df[test_df["label"] == "real"]["still_has_dateline"].mean())


Real articles still containing dateline after stripping:
0.00031456432840515884


### 10.1 Re-evaluate All Three Models on Stripped Text (No Retraining)


In [21]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

stripped_results = []

# --- Baseline (TF-IDF) ---
baseline_stripped_preds = baseline_pipeline.predict(test_df["text_stripped"].astype(str))
acc = accuracy_score(test_df["label_id"], baseline_stripped_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_df["label_id"], baseline_stripped_preds, average="binary")
stripped_results.append({"model": "TF-IDF + Logistic Regression", "stripped_accuracy": acc, "stripped_precision": prec, "stripped_recall": rec, "stripped_f1": f1})
print(f"TF-IDF stripped accuracy: {acc:.4f}")

# --- DistilBERT ---
texts_stripped = test_df["text_stripped"].astype(str).tolist()
all_probs = []
for i in range(0, len(texts_stripped), 32):
    batch = texts_stripped[i:i+32]
    all_probs.append(distilbert_predict_proba(batch))
distilbert_stripped_preds = np.vstack(all_probs).argmax(axis=1)
acc = accuracy_score(test_df["label_id"], distilbert_stripped_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_df["label_id"], distilbert_stripped_preds, average="binary")
stripped_results.append({"model": "DistilBERT", "stripped_accuracy": acc, "stripped_precision": prec, "stripped_recall": rec, "stripped_f1": f1})
print(f"DistilBERT stripped accuracy: {acc:.4f}")

# --- RoBERTa ---
all_probs = []
for i in range(0, len(texts_stripped), 32):
    batch = texts_stripped[i:i+32]
    all_probs.append(roberta_predict_proba(batch))
roberta_stripped_preds = np.vstack(all_probs).argmax(axis=1)
acc = accuracy_score(test_df["label_id"], roberta_stripped_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_df["label_id"], roberta_stripped_preds, average="binary")
stripped_results.append({"model": "RoBERTa", "stripped_accuracy": acc, "stripped_precision": prec, "stripped_recall": rec, "stripped_f1": f1})
print(f"RoBERTa stripped accuracy: {acc:.4f}")


TF-IDF stripped accuracy: 0.9805


DistilBERT stripped accuracy: 0.7283


RoBERTa stripped accuracy: 0.4577


### 10.2 Compare: Original vs Dateline-Stripped Accuracy


In [22]:
original_comparison = pd.read_csv("full_model_comparison.csv")
original_acc = dict(zip(original_comparison["model"], original_comparison["test_accuracy"]))

stripped_df = pd.DataFrame(stripped_results)
stripped_df["original_accuracy"] = stripped_df["model"].map(original_acc)
stripped_df["accuracy_drop"] = stripped_df["original_accuracy"] - stripped_df["stripped_accuracy"]

stripped_df.to_csv("dateline_stripped_comparison.csv", index=False)
stripped_df[["model", "original_accuracy", "stripped_accuracy", "accuracy_drop"]]


,model,original_accuracy,stripped_accuracy,accuracy_drop
0,TF-IDF + Logistic Regression,0.986025,0.980504,0.005521
1,DistilBERT,0.999827,0.728261,0.271567
2,RoBERTa,0.998792,0.457729,0.541063


### 10.3 Interpretation

- **Small drop** (a few percentage points): the model had learned genuine content signal beyond the dateline; the shortcut was a minor boost.
- **Large drop, but still well above 50%**: the model relies heavily on the shortcut, but retains some real signal.
- **Drop toward ~50% (random guessing for a balanced binary task)**: the model had learned almost nothing beyond the dateline shortcut.


### 10.4 Inspect Stripped Text Quality


In [23]:
sample_check = test_df[test_df["label"] == "real"].sample(8, random_state=7)

for _, row in sample_check.iterrows():
    print("ORIGINAL:", row["text"][:150])
    print("STRIPPED:", row["text_stripped"][:150])
    print("-" * 100)


ORIGINAL: WASHINGTON (Reuters) - The White House plan to cut transportation spending, privatize air traffic control and scrap long-distance train service is fac
STRIPPED: The White House plan to cut transportation spending, privatize air traffic control and scrap long-distance train service is facing strong resistance i
----------------------------------------------------------------------------------------------------
ORIGINAL: BOSTON (Reuters) - Donald Trump on Thursday tweeted his support of Maine catalog retailer L.L. Bean after an activist group opposed to the U.S. presid
STRIPPED: Donald Trump on Thursday tweeted his support of Maine catalog retailer L.L. Bean after an activist group opposed to the U.S. president-elect called fo
----------------------------------------------------------------------------------------------------
ORIGINAL: NEW YORK (Reuters) - Two aides in charge of running Hillary Clinton’s presidential campaign were taken aback as news broke in March 2015 of Clin